## Modelagem
### Lógica do Escalonamento Proporcional
Podemos usar o tempo em minutos de cada timeframe como a base do nosso cálculo. A ideia é que o threshold cresça com a raiz quadrada do tempo (já que a volatilidade geralmente escala com raiz de t e o horizonte acompanhe a duração da tendência esperada.

In [0]:
import pyspark.sql.functions as f
from pyspark.sql.window import Window

df_feat = spark.table('workspace.default.tbtc_features').orderBy("timestamp")
timeframe = '1h'

w = Window.orderBy("timestamp")

In [0]:
# horizonte de previsão (3 horas para frente? depende do timeframe?)
# target = 1 se o preço subir > x% em xh, 0 caso contrário

# 1. Definir a base de tempo em minutos para cada timeframe
tf_map = {'15m': 15, '1h': 60, '4h': 240}
minutos_base = tf_map.get(timeframe, 60)

# 2. Definir o "Peso" ou Intensidade (Ajuste apenas aqui!)
# Peso 1.0 = Conservador | Peso 1.5 = Agressivo | Peso 0.8 = Scalper
peso_agressividade = 0.8

# 3. Cálculo Dinâmico
# O Horizonte: Queremos olhar sempre ~4 a 6 horas para frente, idependente do TF
horizonte = int((360 / minutos_base) * peso_agressividade)

# O Threshold: Baseado na volatilidade esperada (raiz quadrada do tempo)
# Multiplicador base de 0.001 (0.1%) por unidade de tempo escalada
import math
threshold = round(0.001 * math.sqrt(minutos_base) * peso_agressividade, 4)

print(f"--- Configuração Dinâmica para {timeframe} ---")
print(f"Horizonte: {horizonte} candles | Alvo: {threshold*100:.2f}%")

df_model = df_feat.withColumn(
    "target",
    f.when(
        (f.lead("close", horizonte).over(w) > f.col("close") * (1 + threshold)), 
        1
    ).otherwise(0)
)

In [0]:
# remove nulos gerados pelas janelas e pelo lead do target
df_model = df_model.dropna()

# converte o sinal categórico para numérico
df_model = df_model.withColumn(
    "vol_signal_idx", 
    f.when(f.col("vol_signal") == "Normal", 0)
    .when(f.col("vol_signal") == "Forte compra (baleia)", 1)
    .otherwise(2) # Forte venda (dump)
)
df_model.limit(5).display()

In [0]:
df_model.groupBy("target").count().show()

| target | count |
|--------|-------|
|   0    |  902  |
|   1    |   97  |

as classes estão desbalanceadas
- threshhold está muito alto?

In [0]:
df_model = df_model.toPandas()

In [0]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

# Momentum de 3 períodos (Variação da variação)
df_model['momentum'] = (df_model['close'] - df_model['close'].shift(3)) / df_model['close'].shift(3)
# Hora do dia (Feature Categórica)
df_model['hour'] = pd.to_datetime(df_model['timestamp']).dt.hour

features_cols = [
    "log_return", "rsi", "vol_change", "z_score_vol", 
    "z_score_close", "assimetria", "curtosi", "dist_VWAP"
    # , "vol_signal_idx"
    ,"momentum", "hour"
]

df_final = df_model[features_cols + ["target", "timestamp"]].dropna().sort_values("timestamp")

# assembler (Transforma colunas em um único vetor 'features')
X = df_final[features_cols]
y = df_final["target"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [0]:
# ordenar por timestamp (garantia)
df_final = df_final.sort_values("timestamp")

# treino e teste
split_idx = int(len(df_final) * 0.8)

train_df = df_final.iloc[:split_idx]
test_df = df_final.iloc[split_idx:]

# undersampling APENAS na base de treino
ones = train_df[train_df["target"] == 1]
zeros = train_df[train_df["target"] == 0]
ratio = len(ones) / len(zeros)
zeros_sampled = zeros.sample(frac=ratio, random_state=42)
train_df_balanced = pd.concat([ones, zeros_sampled]).sort_values("timestamp")

X_train = train_df_balanced[features_cols]
y_train = train_df_balanced["target"]
X_test = test_df[features_cols]
y_test = test_df["target"]

In [0]:
print(train_df.value_counts("target"))
print(test_df.value_counts("target"))


In [0]:
%skip
# 1. Equilibrando a base (Undersampling)
df_ones = train_df.filter(f.col("target") == 1)
df_zeros = train_df.filter(f.col("target") == 0)

# Deixando a proporção 50/50 ou próxima disso
ratio = df_ones.count() / df_zeros.count()
df_zeros_sampled = df_zeros.sample(withReplacement=False, fraction=ratio, seed=56)
train_df = df_ones.union(df_zeros_sampled)

In [0]:
from sklearn.ensemble import RandomForestClassifier
import sklearn.metrics as m

# Configuração e Treinamento do Modelo
rf = RandomForestClassifier(
    n_estimators=200,      # Mais árvores para estabilizar o sinal
    max_depth=5,           # Árvores rasas para evitar Overfitting em base pequena
    max_samples=0.7,       # Cada árvore vê apenas 70% dos dados
    max_features='sqrt',
    random_state=42
)
X_train = train_df[features_cols]
y_train = train_df["target"]
X_test = test_df[features_cols]
y_test = test_df["target"]

model = rf.fit(X_train, y_train)

# Fazendo as previsões
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

# Avaliação das métricas
auc         = m.roc_auc_score(y_test, probabilities)
accuracy    = m.accuracy_score(y_test, predictions)
f1          = m.f1_score(y_test, predictions)
precision   = m.precision_score(y_test, predictions)
recall      = m.recall_score(y_test, predictions)
cm          = m.confusion_matrix(y_test, predictions)

print(f"Área sob a Curva ROC (AUC): {auc:.4f}")
print(f"Acurácia Geral: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print("Confusion Matrix:")
print(cm)

# previsões
pred_df = test_df.copy()
pred_df["prediction"] = predictions
pred_df["probability"] = probabilities
display(pred_df[["timestamp", "target", "prediction", "probability"]].head(10))

- att0: Apesar da acurácia estar aparentemente boa (81%), a curva ROC está dizendo que o modelo não está conseguindo distinguir entre as classes, com um AUC de 50% — equivalente a uma escolha aleatória (como jogar uma moeda para decidir se o mercado vai subir ou descer)

- att1: acuracia abaixou para 49% após undersampling, porém, roc abaixou tambem! (44%)

- att2: acurácia continua 49%, porém curva roc foi para 55%. isso é bacana. ocorreu após utilizar o threshhold dinâmico

In [0]:
# probabilidade da classe 1 (subida)
probabilities = model.predict_proba(X_test)[:, 1]

# em vez de usar o padrão 0.5, vou baixar o threshold
# se o modelo tiver x% de certeza que vai subir, nós consideramos como sinal.
novo_threshold = 0.30 
custom_predictions = (probabilities >= novo_threshold).astype(int)

# recalcular métricas com o novo threshold
from sklearn.metrics import classification_report, confusion_matrix

print(f"--- Métricas com Threshold de {novo_threshold} ---")
print(confusion_matrix(y_test, custom_predictions))
print(classification_report(y_test, custom_predictions))

# AUC deve permanecer a mesma (pois ela avalia a probabilidade, não a classe fixa)
auc = m.roc_auc_score(y_test, probabilities)
print(f"AUC Inalterada: {auc:.4f}")

fuck!

In [0]:
len(predictions)

In [0]:
import pandas as pd
# extraindo a importância
importances = model.feature_importances_
feat_imp = pd.DataFrame(list(zip(features_cols, importances)), 
                        columns=['Feature', 'Importance']).sort_values(by='Importance', ascending=False)
feat_imp